# Fase 4 - Regresión: Predicción de Propinas

**Pregunta de negocio:** ¿Podemos predecir cuánta propina dejará un pasajero?

## Objetivos de este notebook

1. Cargar datos y filtrar solo pagos con tarjeta de crédito (la propina solo se registra para tarjetas)
2. Discutir por qué predecir propinas es mucho más difícil que predecir tarifas
3. Establecer un baseline con la propina media
4. Entrenar modelos: Regresión Lineal, Ridge, Lasso, Random Forest, XGBoost
5. Comparar modelos con métricas estándar
6. Analizar residuos: mucha más varianza que el modelo de tarifa
7. Feature importance: ¿qué predice la propina?
8. Discusión: R² ~ 0.30 es esperable, el comportamiento humano es variable
9. Análisis de error por subgrupo (zona, hora, rango de tarifa)
10. Guardar el mejor modelo

## ¿Por qué la propina es un target difícil?

A diferencia de la tarifa (determinada por una fórmula), la propina depende de:
- La generosidad del pasajero (no observable)
- La calidad percibida del servicio (no observable)
- Normas culturales del pasajero
- Estado de ánimo, prisa, clima...

Esto hace que el "techo" teórico de predicción sea mucho más bajo. Un R² de 0.30
no significa que el modelo sea malo, sino que la variabilidad inherente del
comportamiento humano limita la predicción.

## 1. Configuración e importaciones

In [ ]:
# Conexión a BigQuery
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import joblib
from scipy import stats

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# XGBoost
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Carga de datos y filtrado a tarjetas de crédito

**Importante:** En NYC, la propina solo se registra electrónicamente cuando el pago
es con tarjeta de crédito (payment_type = 1). Los pagos en efectivo muestran
tip_amount = 0, pero eso no significa que no se dejó propina; simplemente no se
registra. Por lo tanto, **filtramos solo pagos con tarjeta** para evitar sesgo.

In [ ]:
data_path = '../../../data/processed/nyc_taxi/features_engineered.parquet'

if os.path.exists(data_path):
    df_all = pd.read_parquet(data_path)
    print(f"Dataset completo cargado: {df_all.shape}")
    
    # Identificar columnas de payment (one-hot encoded)
    payment_cols = [col for col in df_all.columns if col.startswith('payment_')]
    print(f"Columnas de pago: {payment_cols}")
    
    # Filtrar a pagos con tarjeta de crédito
    # En el one-hot encoding con drop='first', la categoría base es '1' (tarjeta)
    # Si todas las payment_X son 0, entonces es tarjeta de crédito (categoría de referencia)
    if payment_cols:
        credit_mask = df_all[payment_cols].sum(axis=1) == 0
        df = df_all[credit_mask].copy()
        # Eliminar columnas de pago (ya no necesarias tras filtrar)
        df = df.drop(columns=payment_cols)
    else:
        df = df_all.copy()

else:
    print("Parquet no encontrado. Cargando desde BigQuery...")
    query = """
    SELECT
        trip_distance,
        TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) / 60.0 AS duration_min,
        CASE
            WHEN TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) > 0 AND trip_distance > 0
            THEN trip_distance / (TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) / 3600.0)
            ELSE NULL
        END AS speed_mph,
        passenger_count,
        EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
        EXTRACT(DAYOFWEEK FROM pickup_datetime) - 1 AS pickup_day_of_week,
        EXTRACT(MONTH FROM pickup_datetime) AS month,
        fare_amount,
        tip_amount,
        CASE WHEN fare_amount > 0 THEN (tip_amount / fare_amount) * 100 ELSE NULL END AS tip_pct
    FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
    WHERE fare_amount BETWEEN 2.5 AND 200
        AND trip_distance BETWEEN 0.1 AND 50
        AND passenger_count BETWEEN 1 AND 6
        AND pickup_location_id IS NOT NULL
        AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) BETWEEN 60 AND 7200
        AND tip_amount >= 0
        AND payment_type = '1'
    ORDER BY RAND()
    LIMIT 200000
    """
    df = bq.query_to_df(query)
    df = df.dropna()
    df['is_weekend'] = (df['pickup_day_of_week'] >= 5).astype(int)
    df['is_rush_hour'] = df['pickup_hour'].apply(lambda h: 1 if (7<=h<=9) or (17<=h<=19) else 0)
    df['is_night'] = df['pickup_hour'].apply(lambda h: 1 if h >= 20 or h <= 6 else 0)
    df['fare_per_mile'] = np.where(df['trip_distance'] > 0, df['fare_amount'] / df['trip_distance'], 0)
    df['fare_per_minute'] = np.where(df['duration_min'] > 0, df['fare_amount'] / df['duration_min'], 0)

print(f"\nDataset filtrado (solo tarjeta de crédito): {df.shape}")
print(f"\nDistribución de tip_amount:")
print(df['tip_amount'].describe().round(2))

In [ ]:
# Visualizar distribución de propinas
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribución de tip_amount
axes[0].hist(df['tip_amount'], bins=80, color='#FF9800', alpha=0.7, edgecolor='white')
axes[0].axvline(df['tip_amount'].mean(), color='red', linestyle='--',
                label=f"Media: ${df['tip_amount'].mean():.2f}")
axes[0].axvline(df['tip_amount'].median(), color='darkblue', linestyle='-',
                label=f"Mediana: ${df['tip_amount'].median():.2f}")
axes[0].set_title('Distribución de Propinas (tarjeta)', fontweight='bold')
axes[0].set_xlabel('Propina (USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_xlim(0, 20)
axes[0].legend()

# tip_pct distribución
axes[1].hist(df['tip_pct'].clip(upper=50), bins=80, color='#4CAF50', alpha=0.7, edgecolor='white')
axes[1].axvline(df['tip_pct'].median(), color='darkblue', linestyle='-',
                label=f"Mediana: {df['tip_pct'].median():.1f}%")
axes[1].set_title('Distribución del Porcentaje de Propina', fontweight='bold')
axes[1].set_xlabel('Propina (%)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.suptitle('Análisis de Propinas - Solo Pagos con Tarjeta', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nViajes con propina = $0: {(df['tip_amount'] == 0).sum():,} ({(df['tip_amount'] == 0).mean()*100:.1f}%)")
print(f"Propina mediana (> $0): ${df.loc[df['tip_amount'] > 0, 'tip_amount'].median():.2f}")

## 3. Preparación de features y target

Usamos **tip_amount** como target. Incluimos **fare_amount** como feature, ya que
la propina frecuentemente se calcula como un porcentaje de la tarifa.

In [ ]:
# Separar features y target
target_col = 'tip_amount'
exclude_cols = ['tip_amount', 'tip_pct']  # tip_pct sería data leakage parcial

feature_cols = [col for col in df.columns if col not in exclude_cols]
X = df[feature_cols].copy()
y = df[target_col].copy()

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nFeatures incluidos: {list(X.columns)}")
print(f"\nTarget (tip_amount):")
print(f"  Media:   ${y.mean():.2f}")
print(f"  Mediana: ${y.median():.2f}")
print(f"  Std:     ${y.std():.2f}")
print(f"  Rango:   [${y.min():.2f}, ${y.max():.2f}]")

print("\nNota: Incluimos fare_amount como feature porque la propina")
print("generalmente se calcula como un porcentaje de la tarifa.")

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Entrenamiento: X={X_train.shape}, y={y_train.shape}")
print(f"Prueba:        X={X_test.shape}, y={y_test.shape}")
print(f"\nTarget en train: media=${y_train.mean():.2f}, std=${y_train.std():.2f}")
print(f"Target en test:  media=${y_test.mean():.2f}, std=${y_test.std():.2f}")

# Escalar para modelos lineales
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Baseline: Predecir la propina media

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    """Calcula R², MAE y RMSE."""
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"  R²:   {r2:.4f}")
    print(f"  MAE:  ${mae:.2f}")
    print(f"  RMSE: ${rmse:.2f}")
    return {'Modelo': model_name, 'R²': r2, 'MAE': mae, 'RMSE': rmse}

results = []

# Baseline
y_pred_baseline = np.full_like(y_test, y_train.mean())

print("=" * 50)
print("BASELINE: Predecir propina media")
print(f"Predicción constante: ${y_train.mean():.2f}")
print("=" * 50)
result = evaluate_model(y_test, y_pred_baseline, 'Baseline (media)')
result['Tiempo (s)'] = 0.0
results.append(result)

## 6. Regresión Lineal

In [ ]:
print("=" * 50)
print("REGRESIÓN LINEAL")
print("=" * 50)

t0 = time.time()
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
train_time = time.time() - t0

result = evaluate_model(y_test, y_pred_lr, 'Regresión Lineal')
result['Tiempo (s)'] = round(train_time, 3)
results.append(result)

# Coeficientes
coefs = pd.Series(lr.coef_, index=feature_cols).sort_values(key=abs, ascending=False)
print(f"\nTop 5 coeficientes:")
for feat, coef in coefs.head(5).items():
    print(f"  {feat:25s}: {coef:+.4f}")

## 7. Ridge Regression

In [ ]:
print("=" * 50)
print("RIDGE REGRESSION - Búsqueda de alpha")
print("=" * 50)

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
ridge_scores = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    cv_scores = cross_val_score(ridge, X_train_scaled, y_train, cv=5, scoring='r2')
    ridge_scores.append({'alpha': alpha, 'mean_r2': cv_scores.mean(), 'std_r2': cv_scores.std()})
    print(f"  alpha={alpha:8.3f} | R² = {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

ridge_df = pd.DataFrame(ridge_scores)
best_alpha_ridge = ridge_df.loc[ridge_df['mean_r2'].idxmax(), 'alpha']

t0 = time.time()
ridge_best = Ridge(alpha=best_alpha_ridge)
ridge_best.fit(X_train_scaled, y_train)
y_pred_ridge = ridge_best.predict(X_test_scaled)
train_time = time.time() - t0

print(f"\nMejor alpha: {best_alpha_ridge}")
result = evaluate_model(y_test, y_pred_ridge, f'Ridge (alpha={best_alpha_ridge})')
result['Tiempo (s)'] = round(train_time, 3)
results.append(result)

## 8. Lasso Regression

In [ ]:
print("=" * 50)
print("LASSO REGRESSION - Búsqueda de alpha")
print("=" * 50)

alphas_lasso = [0.0001, 0.001, 0.01, 0.1, 1.0]
lasso_scores = []

for alpha in alphas_lasso:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    cv_scores = cross_val_score(lasso, X_train_scaled, y_train, cv=5, scoring='r2')
    
    lasso_temp = Lasso(alpha=alpha, max_iter=10000)
    lasso_temp.fit(X_train_scaled, y_train)
    n_nonzero = np.sum(lasso_temp.coef_ != 0)
    
    lasso_scores.append({
        'alpha': alpha, 'mean_r2': cv_scores.mean(),
        'std_r2': cv_scores.std(), 'n_features': n_nonzero
    })
    print(f"  alpha={alpha:8.4f} | R²={cv_scores.mean():.4f} | Features activos: {n_nonzero}/{len(feature_cols)}")

lasso_df = pd.DataFrame(lasso_scores)
best_alpha_lasso = lasso_df.loc[lasso_df['mean_r2'].idxmax(), 'alpha']

t0 = time.time()
lasso_best = Lasso(alpha=best_alpha_lasso, max_iter=10000)
lasso_best.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_best.predict(X_test_scaled)
train_time = time.time() - t0

print(f"\nMejor alpha: {best_alpha_lasso}")
result = evaluate_model(y_test, y_pred_lasso, f'Lasso (alpha={best_alpha_lasso})')
result['Tiempo (s)'] = round(train_time, 3)
results.append(result)

# Features eliminados
lasso_coefs = pd.Series(lasso_best.coef_, index=feature_cols)
eliminated = lasso_coefs[lasso_coefs == 0]
if len(eliminated) > 0:
    print(f"\nFeatures eliminados por Lasso: {list(eliminated.index)}")

## 9. Random Forest Regressor

In [ ]:
print("=" * 50)
print("RANDOM FOREST - Tuning")
print("=" * 50)

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params,
    cv=3,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

t0 = time.time()
rf_grid.fit(X_train, y_train)
train_time = time.time() - t0

print(f"\nMejores parámetros: {rf_grid.best_params_}")
print(f"Mejor R² (CV): {rf_grid.best_score_:.4f}")

y_pred_rf = rf_grid.best_estimator_.predict(X_test)
print("\nEvaluación en test:")
result = evaluate_model(y_test, y_pred_rf, 'Random Forest (tuned)')
result['Tiempo (s)'] = round(train_time, 3)
results.append(result)

## 10. XGBoost Regressor

In [ ]:
print("=" * 50)
print("XGBOOST - GridSearchCV Tuning")
print("=" * 50)

xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
}

xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    xgb_params,
    cv=3,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

t0 = time.time()
xgb_grid.fit(X_train, y_train)
train_time = time.time() - t0

print(f"\nMejores parámetros: {xgb_grid.best_params_}")
print(f"Mejor R² (CV): {xgb_grid.best_score_:.4f}")

y_pred_xgb = xgb_grid.best_estimator_.predict(X_test)
print("\nEvaluación en test:")
result = evaluate_model(y_test, y_pred_xgb, 'XGBoost (tuned)')
result['Tiempo (s)'] = round(train_time, 3)
results.append(result)

## 11. Comparación de modelos

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R²', ascending=False).reset_index(drop=True)

print("=" * 70)
print("COMPARACIÓN DE MODELOS - Predicción de Propina")
print("=" * 70)
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics_info = {
    'R²': '#4CAF50',
    'MAE': '#FF9800',
    'RMSE': '#F44336'
}

for ax, (metric, color) in zip(axes, metrics_info.items()):
    bars = ax.barh(results_df['Modelo'], results_df[metric], color=color, alpha=0.8)
    ax.set_title(f'Comparación: {metric}', fontweight='bold')
    ax.set_xlabel(metric)
    for bar, val in zip(bars, results_df[metric]):
        fmt = f' {val:.4f}' if metric == 'R²' else f' ${val:.2f}'
        ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2, fmt,
                va='center', fontsize=9)

plt.suptitle('Comparación de Modelos - Predicción de Propina',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Comparación con modelo de tarifa
print("\nComparación con el modelo de tarifa (notebook 02):")
print(f"  Tarifa: R² ~ 0.95 (relación determinista)")
print(f"  Propina: R² ~ {results_df.iloc[0]['R²']:.2f} (comportamiento humano)")
print(f"\n  La diferencia refleja la naturaleza fundamentalmente distinta de ambos targets.")

## 12. Análisis de residuos

Esperamos residuos con mucha más varianza que en el modelo de tarifa. Esto
refleja la variabilidad inherente del comportamiento de propinas.

In [ ]:
# Seleccionar mejor modelo
best_model_name = results_df.iloc[0]['Modelo']
print(f"Análisis de residuos para: {best_model_name}")

if 'XGBoost' in best_model_name:
    y_pred_best = y_pred_xgb
    best_model = xgb_grid.best_estimator_
elif 'Random Forest' in best_model_name:
    y_pred_best = y_pred_rf
    best_model = rf_grid.best_estimator_
else:
    y_pred_best = y_pred_lr
    best_model = lr

residuals = y_test.values - y_pred_best

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Residuos vs Predichos
axes[0, 0].scatter(y_pred_best, residuals, alpha=0.05, s=5, color='#FF9800')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_title('Residuos vs Valores Predichos', fontweight='bold')
axes[0, 0].set_xlabel('Propina Predicha (USD)')
axes[0, 0].set_ylabel('Residuo (USD)')
axes[0, 0].set_ylim(-15, 15)

# 2. Distribución de residuos
axes[0, 1].hist(residuals, bins=100, color='#4CAF50', alpha=0.7, edgecolor='white', density=True)
x_range = np.linspace(-15, 15, 100)
axes[0, 1].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()),
                'r-', linewidth=2, label='Normal teórica')
axes[0, 1].set_title('Distribución de Residuos', fontweight='bold')
axes[0, 1].set_xlabel('Residuo (USD)')
axes[0, 1].set_ylabel('Densidad')
axes[0, 1].set_xlim(-15, 15)
axes[0, 1].legend()

# 3. QQ plot
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('QQ Plot de Residuos', fontweight='bold')
axes[1, 0].get_lines()[0].set_markersize(2)

# 4. Predichos vs Reales
axes[1, 1].scatter(y_test, y_pred_best, alpha=0.05, s=5, color='#2196F3')
max_val = max(y_test.max(), y_pred_best.max())
axes[1, 1].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Predicción perfecta')
axes[1, 1].set_title('Valores Reales vs Predichos', fontweight='bold')
axes[1, 1].set_xlabel('Propina Real (USD)')
axes[1, 1].set_ylabel('Propina Predicha (USD)')
axes[1, 1].set_xlim(0, 20)
axes[1, 1].set_ylim(0, 20)
axes[1, 1].legend()

plt.suptitle(f'Análisis de Residuos - {best_model_name} (Propina)',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f"Estadísticas de residuos:")
print(f"  Media:   ${residuals.mean():.4f}")
print(f"  Std:     ${residuals.std():.4f}")
print(f"  Mediana: ${np.median(residuals):.4f}")
print(f"\nComparación con modelo de tarifa:")
print(f"  En tarifa, los residuos son mucho más concentrados alrededor de 0.")
print(f"  Aquí vemos una dispersión mucho mayor, lo cual era esperable.")

## 13. Feature Importance: ¿Qué predice la propina?

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(
        best_model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    importances.plot(kind='barh', ax=ax, color='#FF9800')
    ax.set_title(f'Feature Importance para Propina - {best_model_name}', fontweight='bold')
    ax.set_xlabel('Importancia')
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 features más importantes para predecir propina:")
    for i, (feat, imp) in enumerate(importances.sort_values(ascending=False).head(10).items(), 1):
        print(f"  {i:2d}. {feat:25s}: {imp:.4f} ({imp*100:.1f}%)")
    
    print("\nInterpretación:")
    print("  - fare_amount es probablemente el feature más importante porque")
    print("    la propina se calcula frecuentemente como un % de la tarifa.")
    print("  - trip_distance y duration_min contribuyen indirectamente")
    print("    (correlacionados con fare_amount).")
    print("  - La hora del día y la zona aportan información sobre")
    print("    diferencias de comportamiento entre grupos de pasajeros.")
else:
    coefs_abs = pd.Series(np.abs(best_model.coef_), index=feature_cols).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(10, 8))
    coefs_abs.plot(kind='barh', ax=ax, color='#FF9800')
    ax.set_title('Magnitud de Coeficientes (modelo lineal)', fontweight='bold')
    ax.set_xlabel('|Coeficiente|')
    plt.tight_layout()
    plt.show()

## 14. Discusión: ¿Por qué R² ~ 0.30 es un resultado aceptable?

Un R² de 0.30 puede parecer bajo comparado con el ~0.95 del modelo de tarifa,
pero hay razones fundamentales para esta diferencia:

1. **Naturaleza del target**: La tarifa sigue una fórmula matemática. La propina
   es una decisión humana influenciada por factores no observables.

2. **Variables no disponibles**: No tenemos datos sobre la calidad del servicio,
   el estado de ánimo del pasajero, su origen cultural, ni si el conductor ayudó
   con el equipaje.

3. **Variabilidad inherente**: Incluso pasajeros "similares" dejan propinas
   muy diferentes. Un mismo pasajero puede dejar 15% un día y 25% otro.

4. **Comparación justa**: En ciencias sociales y del comportamiento, R² entre
   0.20 y 0.40 se considera un efecto moderado a fuerte.

El modelo SÍ captura patrones reales (la propina tiende a ser ~18% de la tarifa),
pero no puede predecir las desviaciones individuales de esta tendencia.

## 15. Análisis de error por subgrupo

Examinamos si el modelo funciona mejor o peor para ciertos subgrupos. Esto
puede revelar sesgos o áreas de mejora.

In [ ]:
# Crear DataFrame de análisis
analysis_df = X_test.copy()
analysis_df['y_true'] = y_test.values
analysis_df['y_pred'] = y_pred_best
analysis_df['residual'] = residuals
analysis_df['abs_error'] = np.abs(residuals)

In [ ]:
# 15.1 Error por hora del día
if 'pickup_hour' in analysis_df.columns:
    error_by_hour = analysis_df.groupby('pickup_hour').agg(
        mae=('abs_error', 'mean'),
        mean_tip=('y_true', 'mean'),
        count=('y_true', 'count')
    ).round(3)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    error_by_hour['mae'].plot(kind='bar', ax=axes[0], color='#F44336', alpha=0.7)
    axes[0].set_title('MAE por Hora del Día', fontweight='bold')
    axes[0].set_xlabel('Hora')
    axes[0].set_ylabel('MAE (USD)')
    
    error_by_hour['mean_tip'].plot(kind='bar', ax=axes[1], color='#2196F3', alpha=0.7)
    axes[1].set_title('Propina Promedio por Hora', fontweight='bold')
    axes[1].set_xlabel('Hora')
    axes[1].set_ylabel('Propina Promedio (USD)')
    
    plt.tight_layout()
    plt.show()
    
    print("Error por hora del día:")
    print(error_by_hour.to_string())

In [ ]:
# 15.2 Error por rango de tarifa
if 'fare_amount' in analysis_df.columns:
    fare_col = 'fare_amount'
else:
    fare_col = None

if fare_col:
    bins_fare = [0, 5, 10, 15, 20, 30, 50, 200]
    labels_fare = ['$0-5', '$5-10', '$10-15', '$15-20', '$20-30', '$30-50', '$50+']
    analysis_df['fare_range'] = pd.cut(analysis_df[fare_col], bins=bins_fare, labels=labels_fare)
    
    error_by_fare = analysis_df.groupby('fare_range', observed=True).agg(
        mae=('abs_error', 'mean'),
        mean_tip=('y_true', 'mean'),
        r2=('y_true', lambda x: r2_score(x, analysis_df.loc[x.index, 'y_pred'])
            if len(x) > 1 else np.nan),
        count=('y_true', 'count')
    ).round(3)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    error_by_fare['mae'].plot(kind='bar', ax=axes[0], color='#FF9800', alpha=0.7)
    axes[0].set_title('MAE por Rango de Tarifa', fontweight='bold')
    axes[0].set_xlabel('Rango de Tarifa')
    axes[0].set_ylabel('MAE (USD)')
    axes[0].tick_params(axis='x', rotation=45)
    
    error_by_fare['r2'].plot(kind='bar', ax=axes[1], color='#4CAF50', alpha=0.7)
    axes[1].set_title('R² por Rango de Tarifa', fontweight='bold')
    axes[1].set_xlabel('Rango de Tarifa')
    axes[1].set_ylabel('R²')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print("\nError por rango de tarifa:")
    print(error_by_fare.to_string())
    print("\nObservación: El MAE tiende a crecer con tarifas más altas,")
    print("lo cual tiene sentido porque propinas más altas tienen más variabilidad.")

In [ ]:
# 15.3 Error por zona (si existen columnas de zona)
zone_cols = [col for col in X_test.columns if col.startswith('zone_')]

if zone_cols:
    test_zones = analysis_df[zone_cols].copy()
    test_zones['zone'] = 'Referencia'
    for col in zone_cols:
        zone_name = col.replace('zone_', '')
        test_zones.loc[test_zones[col] == 1, 'zone'] = zone_name
    
    analysis_df['zone'] = test_zones['zone'].values
    
    error_by_zone = analysis_df.groupby('zone').agg(
        mae=('abs_error', 'mean'),
        mean_tip=('y_true', 'mean'),
        count=('y_true', 'count')
    ).sort_values('mae', ascending=False).round(3)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    error_by_zone['mae'].plot(kind='barh', ax=ax, color='#9C27B0', alpha=0.7)
    ax.set_title('MAE de Propina por Zona de Recogida', fontweight='bold')
    ax.set_xlabel('MAE (USD)')
    plt.tight_layout()
    plt.show()
    
    print("\nError por zona:")
    print(error_by_zone.to_string())
else:
    print("No se encontraron columnas de zona para análisis por subgrupo.")
    
    # Alternativa: analizar por weekend vs weekday si existe
    if 'is_weekend' in analysis_df.columns:
        error_by_weekend = analysis_df.groupby('is_weekend').agg(
            mae=('abs_error', 'mean'),
            mean_tip=('y_true', 'mean'),
            count=('y_true', 'count')
        ).round(3)
        error_by_weekend.index = ['Laborable', 'Fin de semana']
        print("\nError por tipo de día:")
        print(error_by_weekend.to_string())

## 16. Guardar el mejor modelo

In [ ]:
model_dir = '../../../models/nyc_taxi'
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'tip_model.joblib')

model_artifacts = {
    'model': best_model,
    'scaler': scaler,
    'feature_names': feature_cols,
    'model_name': best_model_name,
    'metrics': {
        'R2': float(results_df.iloc[0]['R²']),
        'MAE': float(results_df.iloc[0]['MAE']),
        'RMSE': float(results_df.iloc[0]['RMSE'])
    },
    'note': 'Solo para pagos con tarjeta de crédito. R² bajo es esperado para predicción de propinas.'
}

joblib.dump(model_artifacts, model_path)

print(f"Modelo guardado en: {model_path}")
print(f"Modelo: {best_model_name}")
print(f"Métricas:")
print(f"  R²:   {model_artifacts['metrics']['R2']:.4f}")
print(f"  MAE:  ${model_artifacts['metrics']['MAE']:.2f}")
print(f"  RMSE: ${model_artifacts['metrics']['RMSE']:.2f}")
print(f"\nTamaño del archivo: {os.path.getsize(model_path) / 1024 / 1024:.1f} MB")

# Verificar carga
loaded = joblib.load(model_path)
print(f"\nVerificación: modelo cargado correctamente ({loaded['model_name']})")

## Conclusiones

### Resultados principales

1. **R² ~ 0.30 es el resultado esperado**: Predecir propinas es fundamentalmente
   difícil porque depende de factores no observables (generosidad, calidad del
   servicio, normas culturales).

2. **fare_amount es el predictor dominante**: La propina se calcula típicamente
   como un porcentaje de la tarifa (~18%), por lo que conocer la tarifa nos da
   la mejor pista sobre la propina esperada.

3. **La hora y zona aportan información marginal**: Los viajes nocturnos y
   desde aeropuertos tienden a tener propinas ligeramente diferentes.

4. **Residuos con alta varianza**: A diferencia del modelo de tarifa, los residuos
   muestran mucha dispersión, confirmando que el comportamiento humano es difícil
   de modelar con datos de viaje.

5. **Error mayor en tarifas altas**: El MAE crece con la tarifa porque viajes
   más caros tienen propinas absolutas más variables.

### Lecciones aprendidas

- No todos los problemas de ML son igualmente solucionables
- Un R² bajo no significa un modelo malo si el dominio tiene alta variabilidad inherente
- La comparación con el modelo de tarifa ilustra perfectamente la diferencia entre
  predecir procesos deterministas vs comportamiento humano
- Los modelos ensemble (RF, XGBoost) mejoran sobre los lineales, pero la ganancia
  está limitada por la variabilidad irreducible del target

### Próximos pasos

- Fase 5: explorar clasificación (¿dejará propina por encima del 20%?)
- Considerar features adicionales: clima, eventos especiales, zona de destino
- Probar modelos de deep learning con más datos